In [1]:
!pip install librosa
!pip install soundfile
!pip install noisereduce
!pip install pandas

In [2]:
import os
import numpy as np
import librosa
import soundfile as sf
import noisereduce as nr
import pandas as pd

In [3]:
# Load audio
def load_audio(file_path, sr=16000):
    audio, sample_rate = librosa.load(file_path, sr=sr)
    return audio, sample_rate

# Bandpass filter for breathing range
from scipy.signal import butter, lfilter

def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a

def bandpass_filter(data, lowcut=100, highcut=2000, fs=16000):
    b, a = butter_bandpass(lowcut, highcut, fs)
    y = lfilter(b, a, data)
    return y

# Noise reduction
def noise_suppression(audio, sr):
    return nr.reduce_noise(y=audio, sr=sr, prop_decrease=1.0)

# Full preprocessing pipeline
def preprocess_audio(file_path):
    audio, sr = load_audio(file_path)
    filtered = bandpass_filter(audio, fs=sr)
    cleaned = noise_suppression(filtered, sr)
    return cleaned, sr

In [4]:
import os
import pandas as pd
import soundfile as sf
from zipfile import ZipFile

input_dir = "/kaggle/input/datasets/liamgraphics/simulated-ward-dataset"
output_dir = "/kaggle/working/cleaned_audio/"
os.makedirs(output_dir, exist_ok=True)

metadata = []

for file_name in os.listdir(input_dir):
    if file_name.endswith(".wav"):
        file_path = os.path.join(input_dir, file_name)
        cleaned_audio, sr = preprocess_audio(file_path)  # your preprocessing function
        
        output_path = os.path.join(output_dir, file_name)
        sf.write(output_path, cleaned_audio, sr)
        
        metadata.append({
            "file_name": file_name,
            "sample_rate": sr,
            "length_sec": len(cleaned_audio)/sr
        })

# Save metadata CSV (optional, outside ZIP)
metadata_df = pd.DataFrame(metadata)
metadata_csv_path = os.path.join(output_dir, "metadata.csv")
metadata_df.to_csv(metadata_csv_path, index=False)

# Create ZIP file containing only .wav files
zip_path = "/kaggle/working/cleaned_audio.zip"
with ZipFile(zip_path, 'w') as zipf:
    for file_name in os.listdir(output_dir):
        if file_name.endswith(".wav"):  # only include audio files
            file_full_path = os.path.join(output_dir, file_name)
            zipf.write(file_full_path, arcname=file_name)

print(f"Preprocessing complete. Audio files zipped at {zip_path}")

Preprocessing complete. Audio files zipped at /kaggle/working/cleaned_audio.zip
